In [1]:
import os
os.chdir('C:/Users/altai/Desktop/UCR/IV ciclo 2025/Progra 2/Proyectos/Proyecto02/proyecto_shiny_progra2')
print(f"Directorio de Trabajo Actual: {os.getcwd()}")


Directorio de Trabajo Actual (CWD): C:\Users\altai\Desktop\UCR\IV ciclo 2025\Progra 2\Proyectos\Proyecto02\proyecto_shiny_progra2


In [ ]:
#Librerias
from shiny import App, ui, render, reactive
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

# --- Cargar datos ---
base = pd.read_csv("base_de_datos_cantones.csv")

# --- Se Define interfaz (dos columnas: filtros a la izquierda, gráfico a la derecha)---
app_ui = ui.page_fluid(
    ui.h1("Análisis de Desempleo y Pobreza en Cantones de Costa Rica"),
    
    # Panel de navegación con pestañas
    ui.navset_pill(
        ui.nav("Análisis Principal",
            ui.layout_columns(
        # --- Columna izquierda: filtros ---
        ui.card(
            ui.input_select(
                "provincia",
                "Seleccionar provincia:",
                choices=["Todas"] + sorted(base["provincia"].unique().tolist()),
                selected="Todas"
            ),
            ui.input_slider(
                "rango_des",
                "Filtrar por tasa de desempleo abierto (%)",
                min=base["Tasa de desempleo abierto"].min(),
                max=base["Tasa de desempleo abierto"].max(),
                value=(
                    base["Tasa de desempleo abierto"].min(),
                    base["Tasa de desempleo abierto"].max()
                ),
                step=0.1
            )
        ),

        # --- Columna derecha: gráfico ---
        ui.card(
            ui.output_plot("grafico", width="100%", height="500px"),
            ui.output_text("resumen")  # <--- NUEVO: texto debajo del gráfico
        )
    ),

    ui.hr(),
    ui.p(
        ui.strong("Figura 1:"),
        " El gráfico de dispersión muestra la relación entre la tasa de desempleo abierto y el porcentaje de hogares pobres en los cantones de Costa Rica.",
        " Cada punto representa un cantón, y su posición en el plano cartesiano indica su nivel relativo de desempleo (eje X) y pobreza (eje Y).",
        " La línea negra representa una recta de regresión lineal, que sugiere si existe una tendencia general (positiva o negativa) entre ambas variables.",
        " Este gráfico permite identificar cantones con tasas particularmente altas o bajas en ambas dimensiones,así como explorar posibles correlaciones entre desempleo y pobreza."
    )
)

# --- Definir servidor ---
def server(input, output, session):

    @output
    @render.plot
    def grafico():
        # --- Filtrar por provincia ---
        if input.provincia() == "Todas":
            df = base.copy()
        else:
            df = base[base["provincia"] == input.provincia()]

        # --- Filtrar por rango de desempleo ---
        df_filtrado = df[
            (df["Tasa de desempleo abierto"] >= input.rango_des()[0]) &
            (df["Tasa de desempleo abierto"] <= input.rango_des()[1])
        ]

        # --- Crear gráfico ---
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(
            df_filtrado["Tasa de desempleo abierto"],
            df_filtrado["Porcentaje de hogares pobres"],
            color="teal",
            alpha=0.7,
            edgecolors="black"
        )

        # ---Se hace la línea de tendencia ---
        x = df_filtrado["Tasa de desempleo abierto"]
        y = df_filtrado["Porcentaje de hogares pobres"]

        if len(df_filtrado) > 1:
            coef = np.polyfit(x, y, 1)
            tendencia = np.poly1d(coef)
            ax.plot(x, tendencia(x), color="black", linewidth=2, label="Tendencia lineal")
            ax.legend()

            # --- esto es para mostrar el coeficiente de correlación dentro del gráfico ---
            corr = x.corr(y)
            ax.text(
                0.05, 0.95, f"r = {corr:.2f}",
                transform=ax.transAxes,
                fontsize=10,
                color="black",
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.6)
            )

        # --- Etiquetas de cantones ---
        for _, row in df_filtrado.iterrows():
            ax.text(
                row["Tasa de desempleo abierto"],
                row["Porcentaje de hogares pobres"],
                row["Cantón"],
                fontsize=7,
                alpha=0.7
            )

        # --- Estilo ---
        ax.set_xlabel("Tasa de desempleo abierto (%)")
        ax.set_ylabel("Porcentaje de hogares pobres (%)")
        ax.set_title(
            f"Relación entre desempleo y pobreza por cantón\n({input.provincia()})"
        )
        ax.grid(True, linestyle="--", alpha=0.4)
        plt.tight_layout()
        return fig

    # --- Esto es para mostrar debajo del grafico el tipo de la relación que existe entre las variables ---
    @output
    @render.text
    def resumen():
        if input.provincia() == "Todas":
            df = base.copy()
        else:
            df = base[base["provincia"] == input.provincia()]

        df_filtrado = df[
            (df["Tasa de desempleo abierto"] >= input.rango_des()[0]) &
            (df["Tasa de desempleo abierto"] <= input.rango_des()[1])
        ]

        if len(df_filtrado) > 1:
            corr = df_filtrado["Tasa de desempleo abierto"].corr(df_filtrado["Porcentaje de hogares pobres"])
            interpretacion = (
                "Existe una fuerte relación positiva entre desempleo y pobreza."
                if corr > 0.5 else
                "Existe una relación débil o moderada entre desempleo y pobreza."
                if corr > 0.2 else
                "No se observa una relación clara entre desempleo y pobreza."
            )
            return f"Correlación (r): {corr:.2f}. {interpretacion}"
        else:
            return "Insuficientes datos para calcular la correlación."

# --- Crear la app ---
app = App(app_ui, server)

# --- Para ejecutar---
await asyncio.to_thread(run_app, app, port=8010)